# Phase 5: Baseline Demand Forecasting

## Objective

The objective of this notebook is to build baseline machine learning models for retail demand forecasting.

This notebook establishes a benchmark that will later be compared with advanced forecasting models such as XGBoost, LightGBM, Prophet and ARIMA.

### Workflow

1. Load Forecast Dataset
2. Dataset Validation
3. Data Preparation
4. Feature Selection
5. Time-based Train-Test Split
6. Build Baseline Models
7. Compare Model Performance
8. Save Best Baseline Model

In [29]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib

print("Libraries Imported Successfully")

Libraries Imported Successfully


# Load Forecast Dataset

In [30]:
import pandas as pd

forecast = pd.read_csv("../data/processed/merged_dataset.csv")

print("Dataset Loaded Successfully")
forecast.head()

Dataset Loaded Successfully


,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,...,day_name,day_of_week,is_weekend,lag_1,lag_7,lag_30,rolling_7,rolling_30,stock_gap,stock_status
0,2022-09-18,RCPT01376550,ST06,SKU01719,CUST04115,1,460.32,460.32,Online,0.0,...,Sunday,6,1,1.0,2.0,1.0,2.428571,1.900000,0.0,Normal
1,2023-05-26,RCPT02695129,ST05,SKU01551,CUST03306,1,179.55,179.55,In-Store,0.0,...,Friday,4,0,1.0,2.0,3.0,1.142857,1.366667,0.0,Normal
2,2025-08-25,RCPT04093705,ST30,SKU00169,CUST09964,5,95.37,476.85,Online,0.0,...,Monday,0,0,1.0,1.0,1.0,2.571429,2.166667,0.0,Normal
3,2025-01-10,RCPT00287939,ST22,SKU00453,CUST04050,1,382.55,382.55,In-Store,0.0,...,Friday,4,0,1.0,1.0,1.0,2.142857,1.733333,0.0,Normal
4,2025-06-22,RCPT03460988,ST21,SKU00845,CUST00967,2,670.83,1341.66,In-Store,0.0,...,Sunday,6,1,2.0,1.0,3.0,1.428571,1.566667,0.0,Normal


# Dataset Information

In [31]:
print("="*60)
print("Dataset Shape :", forecast.shape)
print("="*60)

forecast.info()

Dataset Shape : (100000, 36)
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 36 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   date               100000 non-null  str    
 1   receipt_id         100000 non-null  str    
 2   store_id           100000 non-null  str    
 3   sku_id             100000 non-null  str    
 4   customer_id        100000 non-null  str    
 5   quantity           100000 non-null  int64  
 6   unit_price         100000 non-null  float64
 7   total_value        100000 non-null  float64
 8   channel            100000 non-null  str    
 9   discount_pct       100000 non-null  float64
 10  promo_id           100000 non-null  str    
 11  sku_name           100000 non-null  str    
 12  category           100000 non-null  str    
 13  subcategory        100000 non-null  str    
 14  cost_price         100000 non-null  float64
 15  brand              100000 non-null

In [32]:
print("Duplicate Rows :", forecast.duplicated().sum())

print("\nMissing Values")

forecast.isnull().sum()

Duplicate Rows : 0

Missing Values


date                     0
receipt_id               0
store_id                 0
sku_id                   0
customer_id              0
quantity                 0
unit_price               0
total_value              0
channel                  0
discount_pct             0
promo_id                 0
sku_name                 0
category                 0
subcategory              0
cost_price               0
brand                    0
stock_on_hand            0
reorder_point            0
safety_stock             0
last_restock_date    79496
year                     0
month                    0
month_name               0
quarter                  0
week                     0
day                      0
day_name                 0
day_of_week              0
is_weekend               0
lag_1                    0
lag_7                    0
lag_30                   0
rolling_7                0
rolling_30               0
stock_gap                0
stock_status             0
dtype: int64

# Convert Date Column

In [33]:
forecast["date"] = pd.to_datetime(forecast["date"])

forecast = forecast.sort_values("date")

print("Date converted successfully")

Date converted successfully


# Remove Unnecessary Columns

The following columns are identifiers or descriptive attributes that are not directly useful for model training.

In [34]:
forecast.drop(columns=[

    "receipt_id",

    "customer_id",

    "promo_id",

    "sku_name",

    "subcategory",

    "brand",

    "month_name",

    "day_name",

    "last_restock_date",

    "stock_status"

], inplace=True, errors="ignore")

# Encode Categorical Variables

In [35]:
encoders = {}

for col in ["store_id", "sku_id", "category", "channel"]:

    le = LabelEncoder()

    forecast[col] = le.fit_transform(forecast[col].astype(str))

    encoders[col] = le

print("Encoding Completed")

Encoding Completed


# Define Target Variable

In [36]:
TARGET = "quantity"

# Feature Selection

In [37]:
FEATURES = [

    "store_id",

    "sku_id",

    "category",

    "channel",

    "unit_price",

    "discount_pct",

    "stock_on_hand",

    "reorder_point",

    "safety_stock",

    "year",

    "month",

    "week",

    "day",

    "day_of_week",

    "is_weekend",

    "lag_1",

    "lag_7",

    "lag_30",

    "rolling_7",

    "rolling_30",

    "stock_gap"

]

# Time-based Train-Test Split

The oldest 80% observations are used for training and the most recent 20% observations are used for testing.

In [38]:
split_index = int(len(forecast) * 0.80)

train = forecast.iloc[:split_index]

test = forecast.iloc[split_index:]

print(train.shape)

print(test.shape)

(80000, 26)
(20000, 26)


# Create Training and Testing Sets

In [39]:
X_train = train[FEATURES]

y_train = train[TARGET]

X_test = test[FEATURES]

y_test = test[TARGET]

# Reduce Training Size

A representative sample is used only for baseline model training to reduce computation time on a very large dataset.

In [40]:
X_train = train[FEATURES]
y_train = train[TARGET]

print(X_train.shape)

(80000, 21)


# Baseline Model 1: Linear Regression

Linear Regression is used as the simplest baseline model to evaluate forecasting performance.

In [41]:
lr = LinearRegression()

lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

print("Linear Regression Training Completed")

Linear Regression Training Completed


# Baseline Model 2: Decision Tree

In [42]:
dt = DecisionTreeRegressor(
    random_state=42
)

dt.fit(X_train, y_train)

dt_pred = dt.predict(X_test)

print("Decision Tree Training Completed")

Decision Tree Training Completed


# Baseline Model 3: Random Forest

In [43]:
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

print("Random Forest Training Completed")

Random Forest Training Completed


# Model Evaluation

In [44]:
def evaluate_model(model_name, y_true, prediction):

    mae = mean_absolute_error(y_true, prediction)

    rmse = np.sqrt(mean_squared_error(y_true, prediction))

    r2 = r2_score(y_true, prediction)

    return {
        "Model": model_name,
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "R2 Score": round(r2, 4)
    }

In [45]:
results = []

results.append(
    evaluate_model(
        "Linear Regression",
        y_test,
        lr_pred
    )
)

results.append(
    evaluate_model(
        "Decision Tree",
        y_test,
        dt_pred
    )
)

results.append(
    evaluate_model(
        "Random Forest",
        y_test,
        rf_pred
    )
)

results_df = pd.DataFrame(results)

results_df

,Model,MAE,RMSE,R2 Score
0,Linear Regression,0.79,0.99,0.1707
1,Decision Tree,1.02,1.44,-0.7582
2,Random Forest,0.81,1.01,0.1340


# Baseline Model Comparison

In [46]:
results_df = results_df.sort_values(
    by="RMSE",
    ascending=True
)

results_df.reset_index(drop=True, inplace=True)

results_df

,Model,MAE,RMSE,R2 Score
0,Linear Regression,0.79,0.99,0.1707
1,Random Forest,0.81,1.01,0.1340
2,Decision Tree,1.02,1.44,-0.7582


# Best Baseline Model

In [47]:
best_model_name = results_df.iloc[0]["Model"]

print("Best Baseline Model :", best_model_name)

Best Baseline Model : Linear Regression


In [48]:
if best_model_name == "Linear Regression":
    best_model = lr

elif best_model_name == "Decision Tree":
    best_model = dt

else:
    best_model = rf

# Save Best Baseline Model

In [49]:
import joblib

joblib.dump(
    best_model,
    "../models/baseline_model.pkl"
)

print("Baseline Model Saved Successfully")

Baseline Model Saved Successfully


In [50]:
print(results_df)

               Model   MAE  RMSE  R2 Score
0  Linear Regression  0.79  0.99    0.1707
1      Random Forest  0.81  1.01    0.1340
2      Decision Tree  1.02  1.44   -0.7582


In [51]:
print(best_model_name)

Linear Regression


# Phase Summary

### Completed Tasks

- Loaded the feature engineered dataset
- Validated dataset quality
- Performed time-based train-test split
- Built three baseline models
    - Linear Regression
    - Decision Tree
    - Random Forest
- Compared model performance using MAE, RMSE and R² Score
- Selected the best-performing baseline model
- Saved the baseline model for future comparison with advanced forecasting models